# D3.6 · Plan, then replan — an investigation that changes its mind

**Function D — The Agentic SOC → Investigate — From an Alert to a Conclusion**

Builds on **[D3.5 · Agent-assisted reconstruction — a timeline you can challenge](https://spbreed.github.io/cyber-commons/lessons/D3.5.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

## What this lesson is

**What it covers.** Investigations that abandon a hypothesis when the evidence refutes it, and keep the abandoned branch visible in the trace.

**Why a security engineer needs it.** Agents score evidence on support, so evidence that supports nothing reads as noise — and refutation is precisely what should force a replan. The result is an investigator that spends the whole incident confirming step one. A conclusion with no visible alternatives also cannot be audited: a reviewer needs to see what was considered and dropped.

## 1 · The hook

An agent forms a hypothesis at step one and spends the rest of the incident finding evidence for it. The evidence that should have stopped it supports nothing at all — which is exactly why an agent scoring only support reads it as noise and carries on.

> **At CyberTravels.** The investigation is CyberTravels' refund incident, and the branch that gets abandoned is the one everybody starts with: the Workflow Agent issued the refund, so the Workflow Agent is the problem. The evidence that kills it is that the agent's own plan for that run contains no refund step — the instruction came from a vendor MCP server's tool description, which is A1.13's risk arriving as an incident.

## 2 · The framework

```
   evidence        supports?        the plan

   1 refund at 03:14   misuse|theft   agent-misuse
   2 agent's session   misuse         agent-misuse
   3 SPIFFE, no human  misuse         agent-misuse
   4 plan has no       --- nothing -- agent-misuse   <- refutes, supports
     refund step                                        nothing, and an
                                                        agent scoring only
                                                        support skips it
   5 vendor tool desc  injection      REPLAN -------> indirect-injection
   6 desc changed      injection      indirect-injection

   the abandoned branch stays in the trace: a reviewer must see that
   agent-misuse was considered and dropped, not that it was never raised
```

Every investigator is wrong at step one. What separates an investigator from an
expensive autocomplete is what happens when the evidence stops fitting.

The way agents fail here is specific. They score evidence on whether it
**supports** a hypothesis — so evidence that supports nothing reads as noise,
when refutation is exactly what should force the replan. The agent keeps
gathering support for a theory the evidence already killed.

> **Anchor → D1.0.** An agent that scores evidence only on support never ends the interval; it spends the whole incident confirming step one. Refutation is what makes the investigation terminate, and the abandoned branch stays in the trace so a reviewer can see it did.

## 3 · The abandoned branch stays in the trace

A reviewer needs to see that a hypothesis was considered and dropped, not that
it was never raised. A conclusion with no visible alternatives looks inevitable,
and an investigation that looks inevitable is one nobody can audit.

In CyberTravels the branch that gets abandoned is the obvious one: the workflow
agent issued the refund, so the workflow agent did it. The evidence that kills
that theory supports nothing at all — the agent's own plan for that run contains
no refund step.

## 4 · An investigation that changes its mind

Six pieces of evidence, three hypotheses, one replan. Step 4 is the one an eager investigator skips.

### The skill — [`skills/secops/investigation-replan-trace/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/secops/investigation-replan-trace/SKILL.md)

```yaml
name: investigation-replan-trace
description: >-
  Run an investigation that must abandon its opening hypothesis, and check the
  replan is visible in the trace. Use when an investigating agent keeps
  confirming its first theory, when evidence that refutes rather than supports
  is being ignored, or when a reviewer needs to see which branches were dropped.
allowed-tools: Read, Grep, Glob
```

# The failure is not being wrong at step one. It is staying wrong.

Every investigator is wrong at step one. What separates an investigator from an
expensive autocomplete is what happens when the evidence stops fitting: a real
one abandons the branch, and the abandoned branch stays in the record.

The specific way agents fail here is subtle. They score evidence on whether it
**supports** a hypothesis, so evidence that supports nothing reads as noise —
when refutation is exactly what should force the replan.

## When to use this

On any investigation an agent drives end to end, and in review of one that
reached a confident conclusion quickly. A trace with no replan on a complex
incident is a warning, not a success.

## Step-by-step

**1 — Record the opening hypothesis explicitly.** An unstated hypothesis cannot
be abandoned, only drifted from.

**2 — Score each piece of evidence for and against, separately.** Support and
refutation are not one axis.

**3 — Trigger a replan on refutation, not on absence of support.** Evidence that
supports nothing is the signal; treat it as a trigger, not as noise.

**4 — Keep the abandoned branch in the trace.** A reviewer must see that a
hypothesis was considered and dropped, not that it was never raised.

**5 — Stop when a hypothesis survives evidence that could have refuted it.**
Not when one accumulates the most support.

## Example

**Input** — six pieces of evidence and three competing hypotheses, in
[`scripts/investigation_replan_trace.py`](scripts/investigation_replan_trace.py).

**Output** — the turn of a real run:

```
4. traces   the agent's plan for that run contains no refund step
   (neutral — refutes nothing, so nothing changes)
5. mcp      the vendor MCP server returned a tool description naming a refund
   REPLAN agent-misuse -> indirect-injection
```

## Output contract

```json
{
  "steps": [{"n": 0, "source": "str", "fact": "str",
             "hypothesis_before": "str", "hypothesis_after": "str",
             "replanned": true}],
  "replans": 0,
  "final": "str"
}
```

## Common edge cases

- **Evidence supports two hypotheses equally.** It is not a discriminator;
  keep both alive rather than picking the first.
- **The refuting evidence arrives first.** Then the opening hypothesis was
  wrong before it was formed, which is fine and should still be recorded.
- **No hypothesis survives.** That is a result. It means the list was
  incomplete, not that the investigation failed.

## Failure modes

- **Anchoring.** Every subsequent query is designed to confirm step one.
- **Silent branch pruning.** The conclusion looks inevitable because the
  alternatives were never written down.
- **Treating "supports nothing" as noise.** It is the strongest signal in the
  trace.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/secops/investigation-replan-trace/scripts/investigation_replan_trace.py
SCRIPT = "skills/secops/investigation-replan-trace/scripts/investigation_replan_trace.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

One replan, from agent-misuse to indirect-injection, with the neutral evidence at step 4 marked as refuting nothing — and the abandoned branch still visible in the trace.

## Your turn

Add evidence that refutes the final hypothesis too. An investigation that cannot end undecided is not investigating.

---

**Next → [D3.7 · Scoping an agentic incident — following the delegation graph](https://spbreed.github.io/cyber-commons/lessons/D3.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D3.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D3.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*